# Analyze Orphanet using ENRICHR

In [1]:
import argparse
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from collections import defaultdict
import os
import requests
import json
import pickle
import time
from common_utils import *

In [2]:
class OrphanetDiseaseEnrichrAnalyzer:
    def __init__(self, ppi_source, input_dir, output_dir, persistence_threshold):
        print(f"\n{'='*80}")
        print(f"INITIALIZING OrphanetDiseaseEnrichrAnalyzer")
        print(f"{'='*80}")
        print(f"PPI Source: {ppi_source}")
        print(f"Input Dir: {input_dir}")
        print(f"Output Dir: {output_dir}")
        
        self.ppi_source = ppi_source
        self.input_dir = input_dir
        self.output_dir = output_dir
        self.cluster_type = 'hidef_from_comparison'
        self.persistence_threshold = persistence_threshold

        
        # Enrichr API endpoints
        self.enrichr_url = 'https://maayanlab.cloud/Enrichr'
        self.add_list_url = f'{self.enrichr_url}/addList'
        self.enrich_url = f'{self.enrichr_url}/enrich'
        
        # Orphanet library name in Enrichr
        self.library_name = 'Orphanet_Augmented_2021'
        
        print(f"\nEnrichr Configuration:")
        print(f"  Base URL: {self.enrichr_url}")
        print(f"  Library: {self.library_name}")
        
        # Setup output directories
        self.cluster_type = 'hidef_from_comparison'
        self.analysis_dir = os.path.join(output_dir, ppi_source, f'orphanet_enrichr_analysis_{self.cluster_type}')
           
        print(f"\nOutput Configuration:")
        print(f"  Analysis directory: {self.analysis_dir}")
        
        ensure_dir(self.analysis_dir)
        print(f"  Directory created/verified: {os.path.exists(self.analysis_dir)}")
        
        # Data containers
        self.clusters = None
        self.seed_genes = None
        self.cluster_seed_mapping = None
        self.results = []
        self.gene_results = []
        
        print(f"{'='*80}\n")

    @property
    def safe_prefix(self):
        """Return a safe prefix for filename generation"""
        return self.cluster_type
    
    def safe_log2(self, value, min_value=1e-10, max_value=1e10):
        """Safely calculate log2 with handling for zero/negative values and infinite results."""
        bounded_value = max(min_value, min(value, max_value))
        return np.log2(bounded_value)
    
    def load_hidef_clusters_multiplex(self, nodes_file, supra_graph_pkl=None,
                                   seed_genes_file=None):
        '''Drop-in replacement for load_hidef_clusters_from_comparison() that
        reads the heterogeneous pipeline's .nodes file instead of the old
        monoplex hidef_communities.pkl. Populates the exact same attributes
        (self.clusters, self.cluster_seed_mapping, self.seed_genes) so nothing
        else in this class needs to change.'''
        import pickle
        from hidef_multiplex_adapter import load_multiplex_clusters

        node_names = None
        if supra_graph_pkl:
            with open(supra_graph_pkl, "rb") as f:
                g = pickle.load(f)
            node_names = g.vs["name"]

        seed_genes = None
        if seed_genes_file:
            seed_genes = set(pd.read_csv(seed_genes_file)["gene"].astype(str))

        self.clusters, self.cluster_seed_mapping, self._multiplex_extra = load_multiplex_clusters(
            nodes_file, node_names=node_names, seed_genes=seed_genes,
            persistence_threshold=self.persistence_threshold,
        )
        self.seed_genes = (
            set().union(*self.cluster_seed_mapping.values())
            if self.cluster_seed_mapping else set()
        )

        if self.clusters:
            sizes = [len(g) for g in self.clusters.values()]
            print(f"Loaded {len(self.clusters)} multiplex clusters "
                  f"(deduped gene-set sizes {min(sizes)}-{max(sizes)}, "
                  f"avg {sum(sizes)/len(sizes):.1f})")
        return bool(self.clusters)
        
    def submit_gene_list_to_enrichr(self, gene_list, description=""):
        """Submit a gene list to Enrichr and return the user list ID."""
        print(f"    Submitting {len(gene_list)} genes to Enrichr...")
        print(f"    First 5 genes: {gene_list[:5]}")
        
        genes_str = '\n'.join(gene_list)
        payload = {
            'list': (None, genes_str),
            'description': (None, description)
        }
        
        try:
            response = requests.post(self.add_list_url, files=payload)
            print(f"    Response status code: {response.status_code}")
            
            if response.ok:
                data = response.json()
                user_list_id = data.get('userListId')
                print(f"    ✓ Successfully submitted. User List ID: {user_list_id}")
                return user_list_id
            else:
                print(f"    ✗ Error submitting gene list: {response.status_code}")
                print(f"    Response text: {response.text[:200]}")
                return None
        except Exception as e:
            print(f"    ✗ Exception submitting gene list: {e}")
            import traceback
            traceback.print_exc()
            return None
    
    def get_enrichment_results(self, user_list_id, library_name):
        """Get enrichment results from Enrichr for a specific library."""
        print(f"    Fetching enrichment results for library: {library_name}")
        query_string = f'?userListId={user_list_id}&backgroundType={library_name}'
        full_url = self.enrich_url + query_string
        print(f"    URL: {full_url}")
        
        try:
            response = requests.get(full_url)
            print(f"    Response status code: {response.status_code}")
            
            if response.ok:
                data = response.json()
                results = data.get(library_name, [])
                print(f"    ✓ Retrieved {len(results)} enrichment results")
                if len(results) > 0:
                    print(f"    Top 3 results:")
                    for i in range(min(3, len(results))):
                        print(f"      {i+1}. {results[i][1] if len(results[i]) > 1 else 'N/A'} (p={results[i][2]:.2e})")
                return results
            else:
                print(f"    ✗ Error getting enrichment results: {response.status_code}")
                print(f"    Response text: {response.text[:200]}")
                return []
        except Exception as e:
            print(f"    ✗ Exception getting enrichment results: {e}")
            import traceback
            traceback.print_exc()
            return []
    
    def analyze_enrichment(self):
        """Analyze disease enrichment in each cluster using Enrichr API."""
        print(f"\n{'='*80}")
        print(f"STARTING ENRICHMENT ANALYSIS")
        print(f"{'='*80}")
        
        if not self.clusters:
            print("ERROR: No clusters loaded!")
            return pd.DataFrame(), pd.DataFrame()
        
        print(f"Total clusters to process: {len(self.clusters)}")
        
        summarized_results = []
        gene_level_results = []
        
        for cluster_idx, (cluster_name, cluster_genes) in enumerate(self.clusters.items()):
            print(f"\n{'-'*80}")
            print(f"Processing cluster {cluster_idx + 1}/{len(self.clusters)}: {cluster_name}")
            print(f"  Cluster size: {len(cluster_genes)} genes")
            
            cluster_genes_list = [str(g) for g in cluster_genes]
            cluster_seed_genes = self.cluster_seed_mapping[cluster_name]
            print(f"  Seed genes: {len(cluster_seed_genes)}")
            
            # Submit gene list to Enrichr
            user_list_id = self.submit_gene_list_to_enrichr(
                cluster_genes_list, 
                description=f"{self.ppi_source}_{cluster_name}"
            )
            
            if not user_list_id:
                print(f"  ✗ Failed to submit gene list for cluster {cluster_name}")
                continue
            
            # Wait a bit to avoid rate limiting
            time.sleep(0.5)
            
            # Get enrichment results
            enrichment_results = self.get_enrichment_results(user_list_id, self.library_name)
            
            if not enrichment_results:
                print(f"  ✗ No enrichment results for cluster {cluster_name}")
                continue
            
            print(f"  Processing {len(enrichment_results)} enrichment results...")
            
            # Keep only the top enriched term per cluster
            # Enrichr returns: [Rank, Term, P-value, Z-score, Combined Score, Overlapping Genes, Adjusted P-value, Old P-value, Old Adjusted P-value]
            if enrichment_results:
                best_result = enrichment_results[0]  # Enrichr returns sorted by combined score
                
                if len(best_result) >= 7:
                    term = best_result[1]
                    p_value = best_result[2]
                    adjusted_p_value = best_result[6]
                    overlapping_genes = best_result[5]
                    combined_score = best_result[4]
                    
                    # Parse overlapping genes - handle both string and list formats
                    if isinstance(overlapping_genes, str):
                        genes_in_term = overlapping_genes.split(';') if overlapping_genes else []
                        overlapping_genes_str = overlapping_genes
                    elif isinstance(overlapping_genes, list):
                        genes_in_term = overlapping_genes
                        overlapping_genes_str = ';'.join(overlapping_genes)
                    else:
                        genes_in_term = []
                        overlapping_genes_str = ""
                    
                    genes_in_term_count = len(genes_in_term)
                    
                    if genes_in_term_count > 0:
                        # Calculate percentage
                        percentage = (genes_in_term_count / len(cluster_genes_list)) * 100
                        
                        # Create standardized result
                        result_dict = create_standardized_results(
                            cluster_name=cluster_name,
                            cluster_genes=cluster_genes,
                            cluster_seed_genes=cluster_seed_genes,
                            cluster_type=self.cluster_type,
                            total_genes_analyzed=genes_in_term_count,
                            percentage=percentage,
                            mean_statistic=combined_score,
                            p_value=p_value,
                            adjusted_p_value=adjusted_p_value,
                            statistic_name='combined_score',
                            genes_in_expression_data=genes_in_term_count
                        )
                        result_dict['disease'] = term
                        result_dict['genes_in_disease'] = genes_in_term_count
                        result_dict['overlapping_genes'] = overlapping_genes_str
                        result_dict['rank'] = best_result[0]
                        summarized_results.append(result_dict)
                        
                        print(f"  ✓ Top enrichment: {term}")
                        print(f"    Combined score: {combined_score:.2f}")
                        print(f"    P-value: {p_value:.2e}")
                        
                        # Create gene-level results
                        for gene in cluster_genes_list:
                            is_seed = gene in cluster_seed_genes
                            
                            gene_level_results.append({
                                'cluster_id': cluster_name,
                                'gene': gene,
                                'is_seed_gene': is_seed,
                                'is_in_expression_data': True,
                                'cluster_combined_score': combined_score,
                                'cluster_p_value': p_value,
                                'cluster_top_disease': term
                            })
        
        print(f"\n{'='*80}")
        print(f"ENRICHMENT ANALYSIS COMPLETE")
        print(f"{'='*80}")
        print(f"Total disease enrichments across all clusters: {len(summarized_results)}")
        print(f"Total gene-level results: {len(gene_level_results)}")
        
        results_df = pd.DataFrame(summarized_results)
        gene_results_df = pd.DataFrame(gene_level_results)
        
        print(f"\nResults DataFrame shape: {results_df.shape}")
        print(f"Gene Results DataFrame shape: {gene_results_df.shape}")
        
        if not results_df.empty:
            # Apply Bonferroni correction
            n_tests = len(results_df)
            results_df['bonferroni_p_value'] = results_df['p_value'] * n_tests
            results_df['bonferroni_p_value'] = results_df['bonferroni_p_value'].clip(upper=1.0)
            print(f"\nBonferroni correction applied: {n_tests} tests")
            
            # Show distribution of diseases per cluster
            diseases_per_cluster = results_df.groupby('cluster_id').size()
            print(f"\nDiseases per cluster statistics:")
            print(f"  Mean: {diseases_per_cluster.mean():.1f}")
            print(f"  Median: {diseases_per_cluster.median():.1f}")
            print(f"  Max: {diseases_per_cluster.max()}")
            print(f"  Min: {diseases_per_cluster.min()}")
        
        if not gene_results_df.empty:
            print(f"\nAnnotating gene results with cluster significance...")
            # Map each gene to whether its cluster has ANY significant disease (using Bonferroni)
            cluster_has_sig_disease = results_df[results_df['bonferroni_p_value'] < 0.05]['cluster_id'].unique()
            gene_results_df['is_cluster_significant'] = gene_results_df['cluster_id'].isin(cluster_has_sig_disease)
            sig_count = gene_results_df['is_cluster_significant'].sum()
            print(f"  {sig_count} genes in clusters with significant disease enrichments (Bonferroni p < 0.05)")
        
        self.results = results_df
        self.gene_results = gene_results_df
        
        print(f"\n{'='*80}\n")
        
        return results_df, gene_results_df
    
    def create_visualizations(self):
        """Create standardized visualizations."""
        print(f"\n{'='*80}")
        print(f"CREATING VISUALIZATIONS")
        print(f"{'='*80}")
        
        plots_dir = os.path.join('plots', self.ppi_source, f'orphanet_enrichr_analysis_{self.cluster_type}_{self.safe_prefix}')
        ensure_dir(plots_dir)
        print(f"Plots directory: {plots_dir}")
        
        if self.results.empty:
            print("✗ No results to visualize")
            return
        
        print(f"Creating visualizations for {len(self.results)} results...")
        
        # 1. Volcano plot
        print("\n1. Creating volcano plot...")
        plt.figure(figsize=(10, 8))
        
        self.results['neg_log_pvalue'] = -np.log10(self.results['p_value'] + 1e-300)
        self.results['log2_combined_score'] = self.results['mean_combined_score'].apply(self.safe_log2)
        
        # Color by significance (Bonferroni corrected)
        colors = ['red' if pval < 0.05 else 'gray' for pval in self.results['bonferroni_p_value']]
        sig_count = sum(1 for c in colors if c == 'red')
        print(f"   Significant results (red, Bonferroni p < 0.05): {sig_count}/{len(colors)}")
        
        plt.scatter(self.results['log2_combined_score'], self.results['neg_log_pvalue'], 
                   c=colors, alpha=0.6, s=100)
        
        plt.axhline(y=-np.log10(0.05), color='black', linestyle='--', alpha=0.5)
        
        plt.xlabel('log2(Combined Score)')
        plt.ylabel('-log10(p-value)')
        plt.title(f'Orphanet Enrichr Analysis - {self.ppi_source}')
        plt.tight_layout()
        
        volcano_path = os.path.join(plots_dir, 'orphanet_enrichr_volcano_plot.png')
        plt.savefig(volcano_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   ✓ Saved: {volcano_path}")
        
        # 2. Top enrichments bar plot
        print("\n2. Creating top enrichments bar plot...")
        if not self.results.empty:
            top_results = self.results.nlargest(20, 'mean_combined_score')
            print(f"   Plotting top {len(top_results)} enrichments")
            
            plt.figure(figsize=(14, 10))
            y_labels = [f"{row['cluster_id'][:30]}: {row['disease'][:50]}" 
                       for _, row in top_results.iterrows()]
            plt.barh(range(len(top_results)), top_results['mean_combined_score'])
            plt.yticks(range(len(top_results)), y_labels, fontsize=8)
            plt.xlabel('Combined Score')
            plt.title(f'Top 20 Orphanet Enrichments - {self.ppi_source}')
            plt.tight_layout()
            
            bar_path = os.path.join(plots_dir, 'top_enrichments.png')
            plt.savefig(bar_path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"   ✓ Saved: {bar_path}")
        
        print(f"\n{'='*80}\n")
    
    def generate_report(self):
        """Generate a simple text report."""
        print(f"\n{'='*80}")
        print(f"GENERATING REPORT")
        print(f"{'='*80}")
        
        report_file = os.path.join(self.analysis_dir, 'analysis_report.txt')
        print(f"Report file: {report_file}")
        
        with open(report_file, 'w') as f:
            f.write(f"Orphanet Enrichr Analysis Report\n")
            f.write(f"=================================\n\n")
            f.write(f"PPI Source: {self.ppi_source}\n")
            f.write(f"Cluster Type: {self.cluster_type}\n")
            f.write(f"Annotation: {self.safe_prefix}\n")
            f.write(f"Enrichr Library: {self.library_name}\n")
            f.write(f"Total clusters: {len(self.clusters)}\n\n")
            
            if not self.results.empty:
                f.write(f"Analysis Results:\n")
                f.write(f"Total disease enrichments: {len(self.results)}\n")
                f.write(f"Significant enrichments (adj p < 0.05): {(self.results['adjusted_p_value'] < 0.05).sum()}\n")
                
                # Diseases per cluster
                diseases_per_cluster = self.results.groupby('cluster_id').size()
                f.write(f"\nDiseases per cluster:\n")
                f.write(f"  Mean: {diseases_per_cluster.mean():.1f}\n")
                f.write(f"  Median: {diseases_per_cluster.median():.1f}\n")
                f.write(f"  Range: {diseases_per_cluster.min()}-{diseases_per_cluster.max()}\n\n")
                
                # Top enrichments
                f.write("Top 20 Enrichments:\n")
                top_results = self.results.nlargest(20, 'mean_combined_score')
                for idx, row in top_results.iterrows():
                    f.write(f"\n  {row['cluster_id']}: {row['disease']}\n")
                    f.write(f"    Combined Score={row['mean_combined_score']:.2f}, ")
                    f.write(f"p={row['p_value']:.2e}, adj_p={row['adjusted_p_value']:.2e}\n")
                    #f.write(f"    Genes: {row['genes_in_disease']}/{row['total_genes_in_cluster']}\n")
        
        print(f"✓ Report saved: {report_file}")
        print(f"\n{'='*80}\n")
    
    def run_analysis(self):
        """Run the complete analysis pipeline."""
        print(f"\n{'#'*80}")
        print(f"# STARTING ORPHANET ENRICHR ANALYSIS")
        print(f"# PPI Source: {self.ppi_source}")
        print(f"# Cluster Type: {self.cluster_type}")
        print(f"{'#'*80}\n")
        
        # Load clusters
        print(f"STEP 1: Loading HiDeF clusters from comparison results...")
        if not self.load_hidef_clusters_multiplex(
            nodes_file='../11_hidef_results_faithful_cpm/heterogeneous_hidef_faithful_cpm.nodes',
            supra_graph_pkl='../11_hidef_hetero/supra_graph.pkl',
            seed_genes_file='../00_inputs/seed_genes.csv'):
            print("✗ Failed to load HiDeF clusters")
            return False

        print(f"✓ Loaded {len(self.clusters)} clusters with {len(self.seed_genes)} seed genes")
        print(f"  Cluster names: {list(self.clusters.keys())[:5]}{'...' if len(self.clusters) > 5 else ''}")
           
        # Run enrichment analysis
        print(f"\nSTEP 2: Running enrichment analysis...")
        results_df, gene_results_df = self.analyze_enrichment()
        
        # Save results
        if not results_df.empty:
            print(f"\nSTEP 3: Saving results...")
            save_results(results_df, gene_results_df, self.analysis_dir, 'orphanet_enrichr')
            print(f"✓ Results saved to: {self.analysis_dir}")
            
            # Create visualizations
            print(f"\nSTEP 4: Creating visualizations...")
            self.create_visualizations()
            
            # Generate report
            print(f"\nSTEP 5: Generating report...")
            self.generate_report()
            
            print(f"\n{'#'*80}")
            print(f"# ANALYSIS COMPLETE!")
            print(f"# Results saved to: {self.analysis_dir}")
            print(f"{'#'*80}\n")
            
            return True
        else:
            print("\n✗ No results generated")
            print(f"  - Clusters loaded: {len(self.clusters) if self.clusters else 0}")
            print(f"  - Results DataFrame empty: {results_df.empty}")
            print(f"  - Gene Results DataFrame empty: {gene_results_df.empty}")
            return False

In [ ]:
ppi_source='combined26_multiplex'
input_dir='./output'
output_dir='./output'
persistence_threshold=5

analyzer = OrphanetDiseaseEnrichrAnalyzer(
    ppi_source=ppi_source,
    input_dir=input_dir,
    output_dir=output_dir,
    persistence_threshold=persistence_threshold
)
analyzer.run_analysis()


INITIALIZING OrphanetDiseaseEnrichrAnalyzer
PPI Source: combined26_multiplex
Input Dir: ./output
Output Dir: ./output

Enrichr Configuration:
  Base URL: https://maayanlab.cloud/Enrichr
  Library: Orphanet_Augmented_2021

Output Configuration:
  Analysis directory: ./output/combined26_multiplex/orphanet_enrichr_analysis_hidef_from_comparison
  Directory created/verified: True


################################################################################
# STARTING ORPHANET ENRICHR ANALYSIS
# PPI Source: combined26_multiplex
# Cluster Type: hidef_from_comparison
################################################################################

STEP 1: Loading HiDeF clusters from comparison results...
[multiplex adapter] 1,581 clusters kept from ../11_hidef_results_faithful_cpm/heterogeneous_hidef_faithful_cpm.nodes (dropped 1 below persistence=5, 6,414 below min_size_after_dedup=2, 0 with unresolved index-form members)
Loaded 1581 multiplex clusters (deduped gene-set sizes 2-13832, 

In [ ]:
pwd